In [0]:
# =============================================================
# Notebook : 01_gold_sales_summary.py
# Purpose  : Silver → Gold for daily sales summary
# Sources  : walmart_silver.fact_pos_transactions
#            walmart_silver.fact_inventory_daily
# Target   : gold/gold_sales_daily_summary/ (Delta)
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

GOLD_PATH = "abfss://gold@walmartdata.dfs.core.windows.net/"

print("Building Gold: Sales Daily Summary...")
spark.sql("CREATE DATABASE IF NOT EXISTS walmart_gold")

In [0]:
# ── Gold Table 1: Daily sales summary by store + category ────
df_sales_summary = spark.sql("""
    SELECT
        sale_date,
        sale_year,
        sale_month,
        store_id,
        store_city,
        store_region,
        payment_method,
        basket_size_category,
        COUNT(*)                            AS total_transactions,
        SUM(total_amount)                   AS total_revenue_inr,
        AVG(total_amount)                   AS avg_basket_size,
        MIN(total_amount)                   AS min_basket,
        MAX(total_amount)                   AS max_basket,
        SUM(tax_amount)                     AS total_tax_collected,
        SUM(item_count)                     AS total_items_sold,
        COUNT(DISTINCT customer_loyalty_id) AS unique_loyalty_customers,
        SUM(CASE WHEN is_loyalty_transaction
                 THEN 1 ELSE 0 END)         AS loyalty_transactions,
        ROUND(SUM(CASE WHEN is_loyalty_transaction
                 THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                AS loyalty_penetration_pct,
        current_timestamp()                 AS gold_processed_at
    FROM walmart_silver.fact_pos_transactions
    GROUP BY
        sale_date, sale_year, sale_month,
        store_id, store_city, store_region,
        payment_method, basket_size_category
    ORDER BY sale_date, total_revenue_inr DESC
""")

print(f"Sales summary rows: {df_sales_summary.count():,}")
display(df_sales_summary.limit(5))

In [0]:
# ── Write Gold Delta ──────────────────────────────────────────
(
    df_sales_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sale_year", "sale_month", "store_region")
    .save(f"{GOLD_PATH}gold_sales_daily_summary/")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_gold.gold_sales_daily_summary
    USING DELTA
    LOCATION '{GOLD_PATH}gold_sales_daily_summary/'
""")

print("✅ walmart_gold.gold_sales_daily_summary registered")

In [0]:
# ── Gold Table 2: Store performance KPIs ─────────────────────
df_store_kpis = spark.sql("""
    SELECT
        p.store_id,
        p.store_city,
        p.store_region,
        COUNT(*)                            AS total_transactions,
        ROUND(SUM(p.total_amount), 2)       AS total_revenue,
        ROUND(AVG(p.total_amount), 2)       AS avg_basket_size,
        COUNT(DISTINCT p.sale_date)         AS active_days,
        ROUND(SUM(p.total_amount)
              / COUNT(DISTINCT p.sale_date), 2) AS avg_daily_revenue,
        SUM(p.item_count)                   AS total_items_sold,
        -- Inventory health join
        COUNT(DISTINCT CASE WHEN i.needs_reorder = true
              THEN i.sku END)               AS skus_needing_reorder,
        COUNT(DISTINCT CASE WHEN i.stockout_risk = 'STOCKOUT'
              THEN i.sku END)               AS stockout_skus,
        current_timestamp()                 AS gold_processed_at
    FROM walmart_silver.fact_pos_transactions p
    LEFT JOIN walmart_silver.fact_inventory_daily i
           ON p.store_id = i.store_id
    GROUP BY p.store_id, p.store_city, p.store_region
    ORDER BY total_revenue DESC
""")

print("\nStore Performance KPIs:")
display(df_store_kpis)

In [0]:
# ── Write store KPIs to Gold ──────────────────────────────────
(
    df_store_kpis
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_PATH}gold_store_kpis/")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_gold.gold_store_kpis
    USING DELTA
    LOCATION '{GOLD_PATH}gold_store_kpis/'
""")

print("✅ walmart_gold.gold_store_kpis registered")